In [12]:
orcid = '0000-0003-4643-8932' # Fill your orcid here

In [13]:
import requests

We use the `/works` api to list all works related to the orcid. This gives a summary of all works, so citation information is not included. We collect the `put-code` of all works to retrieve the citation information later.

In [14]:
response = requests.get('https://pub.orcid.org/v3.0/{}/works'.format(orcid),
                        headers={"Accept": "application/orcid+json" })
record = response.json()

In [20]:
def get_doi(work):
    ids = work.get('external-ids') or {}
    for ext in ids.get('external-id') or []:
        if (ext.get('external-id-type') or '').lower() == 'doi':
            return ext.get('external-id-value')
    return None

citations = []
skipped = []
for put_code in put_codes:
    response = requests.get('https://pub.orcid.org/v3.0/{}/work/{}'.format(orcid, put_code),
                            headers={"Accept": "application/orcid+json"})
    work = response.json()
    title = work['title']['title']['value']

    if work['type'] == 'DATA_SET':
        skipped.append((title, 'dataset'))
        continue

    cit = work.get('citation')
    if cit and cit['citation-type'].upper() == 'BIBTEX':
        citations.append(cit['citation-value'])
        continue

    doi = get_doi(work)
    if doi:
        r = requests.get('https://doi.org/' + doi, headers={'Accept': 'application/x-bibtex'})
        if r.ok:
            r.encoding = 'utf-8'
            citations.append(r.text)
            continue

    skipped.append((title, 'no BibTeX or DOI'))

print(len(citations), 'citations kept')
for title, reason in skipped:
    print('Skipped:', title, '-', reason)

22 citations kept
Skipped: Experimental and Numerical Modelling of Boundary Effects on Internal Solitary Waves - no BibTeX or DOI


We use the `/<orcid>/work/<put-code>` endpoint to retrieve the citation information for each record.

In [22]:
with open('output.bib', 'w') as bibfile:
    for citation in citations:
        bibfile.write(citation)
        bibfile.write('\n')

from google.colab import files
files.download('output.bib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>